<a href="https://colab.research.google.com/github/rahulkate173/Deep-Learning-Quest-/blob/main/DataAugmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import datasets
import transformers
import torch
import numpy as np
import torchvision
print(f"[INFO] transformers version",transformers.__version__)

[INFO] transformers version 5.0.0


In [3]:
from datasets import load_dataset
dataset = load_dataset("mrdbourke/trashify_manual_labelled_images")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00003.parquet:   0%|          | 0.00/334M [00:00<?, ?B/s]

data/train-00001-of-00003.parquet:   0%|          | 0.00/348M [00:00<?, ?B/s]

data/train-00002-of-00003.parquet:   0%|          | 0.00/345M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1128 [00:00<?, ? examples/s]

In [4]:
from transformers import AutoModelForObjectDetection
from transformers import AutoImageProcessor
from typing import List,Tuple
from dataclasses import dataclass , asdict

In [9]:
categories = dataset["train"].features["annotations"]["category_id"]
categories

List(ClassLabel(names=['bin', 'hand', 'not_bin', 'not_hand', 'not_trash', 'trash', 'trash_arm']))

In [10]:
## creating and map
id2label = {key:val for key,val in enumerate(categories.feature.names)}
id2label

{0: 'bin',
 1: 'hand',
 2: 'not_bin',
 3: 'not_hand',
 4: 'not_trash',
 5: 'trash',
 6: 'trash_arm'}

In [11]:
label2id = {val:key for key,val in id2label.items()}
label2id

{'bin': 0,
 'hand': 1,
 'not_bin': 2,
 'not_hand': 3,
 'not_trash': 4,
 'trash': 5,
 'trash_arm': 6}

In [12]:
MODEL_PATH = "PekingU/rtdetr_v2_r50vd"
model = AutoModelForObjectDetection.from_pretrained(
    pretrained_model_name_or_path = MODEL_PATH,
    id2label = id2label,
    label2id = label2id,
    ignore_mismatched_sizes = True # just to finetune with our class
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/172M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie class_embed.0.bias to model.decoder.class_embed.1.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie class_embed.0.weight to model.decoder.class_embed.1.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie class_embed.0.bias to model.decoder.class_embed.2.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie class_embed.0.weight to model.decoder.class_embed.2.weight, but both are present in the checkpoints,

In [13]:
## converting according to our use
IMAGE_SIZE = 640
image_processor = AutoImageProcessor.from_pretrained(
    pretrained_model_name_or_path=MODEL_PATH,
    do_convert_annotations = True, # this will convert the boxes to cx,cy,w,h format
    do_pad = True,
    format = "coco_detection",
    size = {"shortest_edge":IMAGE_SIZE,"longest_edge":IMAGE_SIZE},
    use_fast = True,
    return_segmentation_masks=True
)

preprocessor_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

In [5]:
# class single annotations for single image
@dataclass
class SingleAnnotation:
  category_id : int
  image_id : int
  bbox : List[float]
  is_crowd : int = 0
  area : float = 0.0

  def __post_init__(self):
    if len(self.bbox) != 4:
      raise ValueError("BBox length is not matches")

## image annotation class
@dataclass
class ImageAnnotations:
  image_id : int
  annotations : List[SingleAnnotation]

In [6]:
def format_annotaions(
    image_id : int,
    categories : List[float],
    areas : List[float],
    bbox : List[Tuple[float,float,float,float]]
):
  dict_annotations = [asdict(SingleAnnotation(
    category_id = cat_id,
    image_id = image_id,
    bbox = box,
    area = area
      )) for cat_id,area,box in zip(categories,areas,bbox)]
  return asdict(ImageAnnotations(image_id=image_id,annotations=dict_annotations))

## Okay the new things introduced here is the tv_tensors
### ***tv_tensors*** are the type of tensors are introduced to solve the important problem that is like in augmentation if random flip flipped the image then in the object detection case u have to write the custom function to flip and calculate the bounding boxes and that is solved using tv_tensors

In [26]:
from torchvision import models, datasets, tv_tensors
from torchvision.transforms import v2
transforms_final = v2.Compose(
    [
        v2.ToImage(),
        v2.RandomHorizontalFlip(p=0.5), # Changed to 0.5 for realistic training randomness
        v2.RandomHorizontalFlip(p=1),
        v2.ToDtype(torch.float32, scale=True),
    ]
)

transforms = v2.Compose(
    [
        v2.ToImage(),
        # v2.RandomPhotometricDistort(p=1),
        # v2.RandomZoomOut(fill={tv_tensors.Image: (123, 117, 104), "others": 0}),
        # v2.RandomIoUCrop(),
        v2.RandomHorizontalFlip(p=1),
        v2.SanitizeBoundingBoxes(),
        v2.ToDtype(torch.float32, scale=True),
    ]
)

In [43]:
def preprocess_batch(examples,image_processor,transforms=None):
  """ this function takes examples of image as batch convert the coco annotations and then preprocess
  it with image processor and here if transformations (augmentations) are required that also can add
  and give the final processed batch"""
  images = []
  coco_annotations = []
  for image,image_id,annotations in zip(examples["image"],examples["image_id"],examples["annotations"]):
    print("In loop")
    area = annotations["area"]
    category_id = annotations["category_id"]
    bbox = annotations["bbox"]
    if transforms:
      print("in transform loop")
      h,w = image.size
      tv_img = tv_tensors.Image(image)
      tv_boxes = tv_tensors.BoundingBoxes(
          bbox,
          format = "CXCYWH",
          canvas_size=(h, w)
      )
      tv_labels = torch.tensor(category_id,dtype=torch.int64)
      print("before transforms")
      tv_img, tv_boxes, tv_labels = transforms(tv_img, tv_boxes, tv_labels) # sugmentation pipeline
      print("transform done ")
      # unpacking item back
      image = tv_img
      bbox = tv_boxes.tolist()
      category_id = tv_labels.tolist()
      area = [float(b * b) for b in bbox[0]]
      print(bbox,category_id,area)
      print("===========================")
    print("out form transform")
    annotated_image = format_annotaions(image_id,category_id,area,bbox)
    images.append(image)
    coco_annotations.append(annotated_image)
  processed_batch = image_processor.preprocess(images=images,annotations=coco_annotations,return_tensors="pt")
  print("processing done")
  return processed_batch

In [44]:
group_sample = dataset["train"][0:3]

In [45]:
preprocessed_samples = preprocess_batch(examples=group_sample,
                                        image_processor=image_processor,transforms=transforms_final)

preprocessed_samples.keys()

In loop
in transform loop
before transforms
transform done 
[[756.2999877929688, 545.0999755859375, 402.79998779296875, 336.1000061035156], [1269.5999755859375, 163.6999969482422, 943.4000244140625, 1101.9000244140625]] [1, 0] [571989.6715356447, 297133.98338378966, 162247.83016601577, 112963.21410278324]
out form transform
In loop
in transform loop
before transforms
transform done 
[[280.29998779296875, 653.0, 460.1000061035156, 389.6000061035156], [598.2000122070312, 604.2999877929688, 352.1000061035156, 336.79998779296875], [219.0999755859375, 348.1000061035156, 470.20001220703125, 461.0], [514.9000244140625, 363.1000061035156, 226.0, 312.6000061035156], [788.2000122070312, 194.89999389648438, 163.1999969482422, 285.1000061035156]] [5, 1, 0, 0, 2] [78568.08315673843, 426409.0, 211692.01561645512, 151788.1647558594]
out form transform
In loop
in transform loop
before transforms
transform done 
[[349.5, 677.2000122070312, 210.89999389648438, 376.0], [483.0, 769.7000122070312, 239.6999

KeysView({'pixel_mask': tensor([[[1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         ...,
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1]],

        [[1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         ...,
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1]],

        [[1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         ...,
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1]]]), 'pixel_values': tensor([[[[0.0021, 0.0022, 0.0023,  ..., 0.0039, 0.0039, 0.0039],
          [0.0020, 0.0022, 0.0023,  ..., 0.0039, 0.0039, 0.0039],
          [0.0020, 0.0021, 0.0022,  ..., 0.0039, 0.0039, 0.0039],
          ...,
          [0.0024, 0.0024, 0.0024,  ..., 0.0012, 0.0013, 0.0013],
  

In [38]:
group_sample

{'image': [<PIL.Image.Image image mode=RGB size=960x1280>,
  <PIL.Image.Image image mode=RGB size=960x1280>,
  <PIL.Image.Image image mode=RGB size=960x1280>],
 'image_id': [292, 1053, 1043],
 'annotations': [{'file_name': ['00347467-13f1-4cb9-94aa-4e4369457e0c.jpeg',
    '00347467-13f1-4cb9-94aa-4e4369457e0c.jpeg'],
   'image_id': [292, 292],
   'category_id': [1, 0],
   'bbox': [[523.7000122070312,
     545.0999755859375,
     402.79998779296875,
     336.1000061035156],
    [10.399999618530273,
     163.6999969482422,
     943.4000244140625,
     1101.9000244140625]],
   'iscrowd': [0, 0],
   'area': [135381.078125, 1039532.4375]},
  {'file_name': ['005ae811-9732-40e9-9d4d-7f4e8ce57564.jpeg',
    '005ae811-9732-40e9-9d4d-7f4e8ce57564.jpeg',
    '005ae811-9732-40e9-9d4d-7f4e8ce57564.jpeg',
    '005ae811-9732-40e9-9d4d-7f4e8ce57564.jpeg',
    '005ae811-9732-40e9-9d4d-7f4e8ce57564.jpeg'],
   'image_id': [1053, 1053, 1053, 1053, 1053],
   'category_id': [5, 1, 0, 0, 2],
   'bbox': [[280

In [46]:
preprocessed_samples = preprocess_batch(examples=group_sample,
                                        image_processor=image_processor,transforms=None)

preprocessed_samples.keys()

In loop
out form transform
In loop
out form transform
In loop
out form transform
processing done


KeysView({'pixel_mask': tensor([[[1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         ...,
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1]],

        [[1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         ...,
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1]],

        [[1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         ...,
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1]]]), 'pixel_values': tensor([[[[0.9961, 0.9961, 0.9961,  ..., 0.5922, 0.5569, 0.5255],
          [0.9961, 1.0000, 1.0000,  ..., 0.5765, 0.5529, 0.5216],
          [0.9961, 1.0000, 1.0000,  ..., 0.5725, 0.5490, 0.5137],
          ...,
          [0.3255, 0.3176, 0.3137,  ..., 0.6157, 0.6196, 0.6157],
  